<a href="https://colab.research.google.com/github/Milo443/Deeplearning/blob/main/MPL_sklearn_Featuring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Clasificación de Vehículos con MLPClassifier basado en Extracción de Características (Featuring)

Este cuaderno implementa un modelo de aprendizaje profundo basado en el **Perceptrón Multicapa (`MLPClassifier` de scikit-learn)** adaptado al conjunto de datos extraído mediante técnicas de Visión Artificial: **`DataFrame_Featuring (1).csv`**.

---

### ¿En qué se diferencia este enfoque del "Flattening" (Aplanamiento de píxeles)?
* **Enfoque Flattening (Notebook anterior):** Toma las imágenes en escala de grises y aplana directamente los píxeles a un vector unidimensional (1200 valores de intensidad). Esto conlleva alta sensibilidad al ruido, cambios drásticos por iluminación, rotaciones y desplazamientos espaciales.
* **Enfoque Featuring (Este cuaderno):** Utiliza un vector de **1,184 características de ingeniería visual** compuestas por múltiples descriptores clásicos de visión computacional:
  1. **Canny (1024 características):** Detección de bordes y contornos geométricos.
  2. **LBP - Local Binary Patterns (26 características):** Textura superficial y micro-patrones locales.
  3. **DCT - Discrete Cosine Transform (64 características):** Distribución de frecuencias espaciales.
  4. **ORB (32 características):** Puntos clave invariantes a rotación y escala.
  5. **HSV (24 características):** Distribución y momentos en el espacio de color.
  6. **Hu Moments (7 características):** Momentos invariantes de forma (escala, rotación y traslación).
  7. **LoG - Laplacian of Gaussian (4 características):** Detección de manchas (blobs) multiescala.
  8. **Harris (3 características):** Detección y respuesta de esquinas salientes.

---

### Objetivos del Notebook:
1. **Compatibilidad Total:** Conexión y montaje automático de **Google Drive** y búsqueda inteligente de archivos tanto en la nube (Colab) como en entornos locales.
2. **Carga y Análisis Exploratorio de Datos (EDA):** Visualización de la distribución de las 10 clases de vehículos y familias de descriptores.
3. **Preprocesamiento y Normalización:** Escalado robusto (`StandardScaler`) crucial para descriptores con órdenes de magnitud heterogéneos.
4. **Cálculo Riguroso de Parámetros:** Función para calcular y auditar la cantidad exacta de parámetros entrenables (pesos y sesgos) de cada modelo MLP.
5. **Exploración Sistemática de Hiperparámetros:** Búsqueda y comparación entre diversas arquitecturas ocultas, funciones de activación (`relu`, `tanh`, `logistic`), optimizadores (`adam`, `sgd`) y regularizadores `alpha`.
6. **Entrenamiento del Modelo con los Mejores Hiperparámetros:** Configuración optimizada con Early Stopping para máxima generalización.
7. **Evaluación Integral y Gráficos Detallados:** Curva de pérdida/aprendizaje, matriz de confusión normalizada, métricas por clase (Precision, Recall, F1) e interpretabilidad de pesos de la red.


## 1. Conexión y Montaje de Google Drive (Google Colab)

Si estás ejecutando este cuaderno en **Google Colab**, la siguiente celda montará tu Google Drive para acceder a la carpeta donde tienes guardado el dataset (por ejemplo: `/content/drive/MyDrive/DeepLearning/`). Si estás en un entorno local, el bloque detectará automáticamente que no es Colab y continuará sin errores.


In [ ]:
# Montar Google Drive si se ejecuta en Google Colab
try:
    from google.colab import drive
    import os
    if not os.path.exists('/content/drive/MyDrive'):
        print("Montando Google Drive...")
        drive.mount('/content/drive')
        print("✓ Google Drive montado exitosamente en /content/drive")
    else:
        print("✓ Google Drive ya se encuentra montado.")
except ImportError:
    print("ℹ Entorno local detectado (no es Google Colab). Se utilizarán rutas locales.")


## 2. Importación de Librerías y Configuración del Entorno


In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de scikit-learn
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support
)

# Configuración estética para gráficos de alta calidad
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

# Semilla para máxima reproducibilidad
SEED = 42
np.random.seed(SEED)

def contar_parametros_mlp(modelo):
    """Calcula y desglosa la cantidad exacta de parámetros entrenables del MLP (Pesos W y Sesgos b)."""
    num_pesos = sum(w.size for w in modelo.coefs_)
    num_sesgos = sum(b.size for b in modelo.intercepts_)
    total_params = num_pesos + num_sesgos
    return total_params, num_pesos, num_sesgos

print("Librerías importadas y función de conteo de parámetros configurada correctamente.")


## 3. Carga Inteligente del Dataset (`DataFrame_Featuring`)

El siguiente código:
1. Importa defensivamente `pandas` y `os` para evitar errores de tipo `NameError: name 'pd' is not defined`.
2. Busca el archivo en las rutas habituales de **Google Drive** (`/content/drive/MyDrive/DeepLearning/...`), en el almacenamiento temporal de **Colab** (`/content/...`) y en el **directorio local**.
3. Realiza una búsqueda recursiva automática (`glob`) en caso de que el archivo se encuentre en una subcarpeta distinta de Drive.


In [ ]:
import os
import glob
import pandas as pd

# Rutas potenciales del archivo CSV (Google Drive, Colab y entorno local)
posibles_rutas = [
    '/content/drive/MyDrive/DeepLearning/DataFrame_Featuring (1).csv',
    '/content/drive/MyDrive/Deeplearning/DataFrame_Featuring (1).csv',
    '/content/drive/MyDrive/Deep Learning/DataFrame_Featuring (1).csv',
    '/content/drive/MyDrive/DataFrame_Featuring (1).csv',
    '/content/DataFrame_Featuring (1).csv',
    'DataFrame_Featuring (1).csv',
    './DataFrame_Featuring (1).csv',
    '../DataFrame_Featuring (1).csv'
]

ruta_encontrada = None
for r in posibles_rutas:
    if os.path.exists(r):
        ruta_encontrada = r
        break

# Búsqueda dinámica automática recursiva si no se encuentra en las rutas fijas
if ruta_encontrada is None:
    print("Buscando recursivamente 'DataFrame_Featuring (1).csv' en Google Drive y entorno local...")
    candidatos = (
        glob.glob('/content/drive/MyDrive/**/DataFrame_Featuring (1).csv', recursive=True) +
        glob.glob('/content/**/DataFrame_Featuring (1).csv', recursive=True) +
        glob.glob('**/DataFrame_Featuring (1).csv', recursive=True)
    )
    if candidatos:
        ruta_encontrada = candidatos[0]

if ruta_encontrada is None:
    raise FileNotFoundError(
        "No se encontró 'DataFrame_Featuring (1).csv'. "
        "Asegúrate de: 1) Montar Google Drive ejecutando la celda anterior, o "
        "2) Subir el archivo a /content/ o a /content/drive/MyDrive/DeepLearning/"
    )

print(f"✓ Archivo encontrado exitosamente:")
print(f"  Ruta: {ruta_encontrada}
")

df = pd.read_csv(ruta_encontrada)

print(f"✓ Dataset cargado con éxito:")
print(f" • Muestras totales (filas): {df.shape[0]}")
print(f" • Columnas totales: {df.shape[1]} (1,184 características + 1 etiqueta)")
print(f" • Valores nulos totales: {df.isnull().sum().sum()}")
df.head(5)


## 4. Análisis Exploratorio de Datos (EDA)

Examinamos la distribución de las 10 categorías de vehículos y la composición del vector de características extraídas.


In [ ]:
# Identificar la columna objetivo (asumida como la última columna)
columna_etiqueta = df.columns[-1]
clases_conteo = df[columna_etiqueta].value_counts()

# Desglosar las familias de características según su prefijo
prefijos_descriptores = {}
for col in df.columns[:-1]:
    pref = col.split('_')[0]
    prefijos_descriptores[pref] = prefijos_descriptores.get(pref, 0) + 1

# Crear visualizaciones gemelas: Distribución de Clases y Composición de Descriptores
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Distribución de clases de vehículos
colores_clases = sns.color_palette("mako", len(clases_conteo))
barras = axes[0].bar(clases_conteo.index, clases_conteo.values, color=colores_clases, edgecolor='black', alpha=0.85)
axes[0].set_title("Distribución de Clases de Vehículos en el Dataset")
axes[0].set_xlabel("Tipo de Vehículo")
axes[0].set_ylabel("Cantidad de Muestras")
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)

# Anotar valores sobre las barras
for bar in barras:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2.0, yval + 10, f"{int(yval)}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# Gráfico 2: Desglose de características extraídas por técnica de visión
nombres_descriptores = list(prefijos_descriptores.keys())
cantidades_descriptores = list(prefijos_descriptores.values())
colores_desc = sns.color_palette("viridis", len(nombres_descriptores))

barras_desc = axes[1].barh(nombres_descriptores, cantidades_descriptores, color=colores_desc, edgecolor='black', alpha=0.85)
axes[1].set_title("Composición del Vector de Características (1,184 dims)")
axes[1].set_xlabel("Número de Características")
axes[1].set_ylabel("Familia de Descriptor")
axes[1].grid(axis='x', linestyle='--', alpha=0.5)

for bar in barras_desc:
    xval = bar.get_width()
    axes[1].text(xval + 15, bar.get_y() + bar.get_height()/2.0, f"{int(xval)} ({xval/sum(cantidades_descriptores)*100:.1f}%)",
                 va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("
Resumen de familias de características:")
for k, v in sorted(prefijos_descriptores.items(), key=lambda x: x[1], reverse=True):
    print(f"  • {k.upper():<7}: {v:>4} características ({v/sum(cantidades_descriptores)*100:.2f}%)")


## 5. Preprocesamiento: Separación, Codificación y Normalización

### Importancia Crítica del Escalado en Descriptores Visuales:
En el dataset `DataFrame_Featuring`, cada familia de descriptores posee magnitudes y unidades radicalmente distintas:
* **Hu Moments:** Valores en órdenes de magnitud de $10^{-3}$ a $10^{-9}$.
* **Canny:** Conteo de intensidades de bordes en rangos $0 - 255$.
* **LBP:** Frecuencias de histograma local.
* **DCT:** Coeficientes de frecuencias espaciales positivos y negativos.

Si no aplicamos **`StandardScaler`**, los descriptores con escalas numéricas grandes dominarán completamente la función de costo y el cálculo del gradiente, impidiendo que la red aprenda de descriptores de textura o forma como Hu y LBP.


In [ ]:
# 1. Separar características (X) y etiquetas (y)
X = df.iloc[:, :-1].values
y_raw = df.iloc[:, -1].values

# 2. Codificar etiquetas categóricas a valores numéricos enteros
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_raw)
clases_nombres = label_encoder.classes_

print(f"Clases codificadas ({len(clases_nombres)} categorías):")
for idx, clase in enumerate(clases_nombres):
    print(f"  [{idx}] -> {clase}")

# 3. División estratificada (80% Entrenamiento, 20% Prueba)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=SEED,
    stratify=y_encoded
)

print(f"
Conjunto de Entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de Prueba:        {X_test.shape[0]} muestras")

# 4. Ajuste y transformación con StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Visualizar el efecto del escalado en las características
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.boxplot(X_train[:, :10], tick_labels=[f"F_{i}" for i in range(10)])
ax1.set_title("Distribución de Características ANTES del Escalado (Original)")
ax1.set_ylabel("Valor original")
ax1.grid(True, linestyle='--', alpha=0.5)

ax2.boxplot(X_train_scaled[:, :10], tick_labels=[f"F_{i}" for i in range(10)])
ax2.set_title("Distribución de Características DESPUÉS de StandardScaler (μ=0, σ=1)")
ax2.set_ylabel("Valor normalizado (Z-score)")
ax2.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## 6. Experimentación Sistemática de Hiperparámetros y Auditoría de Parámetros

Evaluamos un conjunto representativo de configuraciones clave de `MLPClassifier` variando:
1. **Arquitectura (`hidden_layer_sizes`):**
   * Redes de 1 capa: `(128,)`, `(256,)`, `(512,)`
   * Redes profundas de 2 capas: `(256, 128)`, `(512, 256)`
   * Redes profundas de 3 capas: `(512, 256, 128)`
2. **Funciones de Activación (`activation`):**
   * `'relu'` (Rectified Linear Unit, estándar y rápida)
   * `'tanh'` (Tangente hiperbólica, rango centrado en 0)
   * `'logistic'` (Sigmoide)
3. **Optimizador (`solver`):**
   * `'adam'` (Gradiente estocástico adaptativo con momentos)
   * `'sgd'` (Con regularización e impulso Nesterov)
4. **Regularización L2 (`alpha`):**
   * Controla la penalización de pesos para mitigar sobreajuste en 1,184 variables.

> **Auditoría de Parámetros del Modelo:** Para cada arquitectura calculamos exactamente la cantidad de parámetros entrenables (pesos y sesgos) y su impacto en precisión y tiempo de ejecución.


In [ ]:
# Definir una batería rigurosa y diversa de configuraciones
configuraciones_experimento = [
    # Experimentos de Arquitectura con ReLU + Adam
    {"nombre": "Cfg 1: Monocapa (128)",       "hls": (128,),           "act": "relu",     "solv": "adam", "alpha": 0.0001, "lr": 0.001},
    {"nombre": "Cfg 2: Monocapa (256)",       "hls": (256,),           "act": "relu",     "solv": "adam", "alpha": 0.0001, "lr": 0.001},
    {"nombre": "Cfg 3: Monocapa (512)",       "hls": (512,),           "act": "relu",     "solv": "adam", "alpha": 0.0005, "lr": 0.001},
    {"nombre": "Cfg 4: Bicapa (256, 128)",    "hls": (256, 128),       "act": "relu",     "solv": "adam", "alpha": 0.001,  "lr": 0.001},
    {"nombre": "Cfg 5: Bicapa (512, 256)",    "hls": (512, 256),       "act": "relu",     "solv": "adam", "alpha": 0.001,  "lr": 0.001},
    {"nombre": "Cfg 6: Tricapa (512, 256, 128)","hls": (512, 256, 128),"act": "relu",    "solv": "adam", "alpha": 0.001,  "lr": 0.001},

    # Experimentos con Función de Activación Tanh
    {"nombre": "Cfg 7: Tanh Monocapa (256)",  "hls": (256,),           "act": "tanh",     "solv": "adam", "alpha": 0.001,  "lr": 0.001},
    {"nombre": "Cfg 8: Tanh Bicapa (512, 256)","hls": (512, 256),     "act": "tanh",     "solv": "adam", "alpha": 0.001,  "lr": 0.001},

    # Experimentos con Activación Logística (Sigmoide)
    {"nombre": "Cfg 9: Logistic Monocapa (256)","hls": (256,),         "act": "logistic", "solv": "adam", "alpha": 0.001,  "lr": 0.001},

    # Experimentos con Optimizador SGD
    {"nombre": "Cfg 10: SGD Momentum (256, 128)","hls": (256, 128),   "act": "relu",     "solv": "sgd",  "alpha": 0.001,  "lr": 0.01}
]

resultados_experimentos = []

print("="*105)
print(f"{'Configuración':<28} | {'Arquitectura':<16} | {'Activ':<8} | {'Parámetros':<11} | {'Accuracy':<9} | {'Iter':<5} | {'Tiempo':<6}")
print("="*105)

for cfg in configuraciones_experimento:
    t_inicio = time.time()
    
    modelo_exp = MLPClassifier(
        hidden_layer_sizes=cfg["hls"],
        activation=cfg["act"],
        solver=cfg["solv"],
        alpha=cfg["alpha"],
        learning_rate_init=cfg["lr"],
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=10,
        random_state=SEED,
        verbose=False
    )
    
    modelo_exp.fit(X_train_scaled, y_train)
    t_fin = time.time()
    
    y_pred_exp = modelo_exp.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_exp)
    duracion = t_fin - t_inicio
    
    total_params, pesos, sesgos = contar_parametros_mlp(modelo_exp)
    
    print(f"{cfg['nombre']:<28} | {str(cfg['hls']):<16} | {cfg['act']:<8} | {total_params:>11,d} | {acc*100:>7.2f}% | {modelo_exp.n_iter_:>4} | {duracion:>5.1f}s")
    
    resultados_experimentos.append({
        "Configuración": cfg["nombre"],
        "Capas": str(cfg["hls"]),
        "Activación": cfg["act"],
        "Solver": cfg["solv"],
        "Alpha": cfg["alpha"],
        "Parámetros": total_params,
        "Pesos": pesos,
        "Sesgos": sesgos,
        "Accuracy": acc,
        "Iteraciones": modelo_exp.n_iter_,
        "Tiempo_s": duracion,
        "Perdida_Final": modelo_exp.loss_
    })

print("="*105)
df_resultados = pd.DataFrame(resultados_experimentos).sort_values(by="Accuracy", ascending=False)


## 7. Visualización Comparativa de Hiperparámetros y Complejidad del Modelo

Analizamos visualmente qué arquitecturas y funciones de activación alcanzan el mejor desempeño y cómo se correlaciona la cantidad de parámetros con la precisión y el tiempo.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Gráfico 1: Comparativa global de precisión ordenada con cantidad de parámetros
df_ordenado = df_resultados.sort_values(by="Accuracy", ascending=True)
paleta = sns.color_palette("crest", len(df_ordenado))

barras = axes[0].barh(df_ordenado["Configuración"], df_ordenado["Accuracy"] * 100, color=paleta, edgecolor='black', alpha=0.85)
axes[0].set_title("Comparación de Precisión (Accuracy %) entre Hiperparámetros", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Precisión en Test (%)")
axes[0].set_xlim(0, 100)
axes[0].grid(axis='x', linestyle='--', alpha=0.6)

for bar, (_, row) in zip(barras, df_ordenado.iterrows()):
    w = bar.get_width()
    axes[0].text(w + 1.0, bar.get_y() + bar.get_height()/2.0, f"{w:.2f}% ({row['Parámetros']:,} params)", va='center', fontsize=8.5, fontweight='bold')

# Gráfico 2: Relación entre Precisión y Cantidad de Parámetros
scatter = axes[1].scatter(
    df_resultados["Parámetros"] / 1000, # Miles de parámetros
    df_resultados["Accuracy"] * 100,
    s=df_resultados["Tiempo_s"] * 15 + 80,
    c=df_resultados["Accuracy"] * 100,
    cmap="plasma",
    edgecolors="black",
    alpha=0.9
)
axes[1].set_title("Compromiso Precisión vs Complejidad del Modelo
(Tamaño del punto proporcional al tiempo de entrenamiento)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Miles de Parámetros Entrenables (k)")
axes[1].set_ylabel("Precisión en Test (%)")
axes[1].grid(True, linestyle='--', alpha=0.6)

for _, row in df_resultados.iterrows():
    axes[1].annotate(
        row["Configuración"].split(":")[1].strip(),
        (row["Parámetros"] / 1000, row["Accuracy"] * 100),
        textcoords="offset points",
        xytext=(0, 8),
        ha='center',
        fontsize=8,
        fontweight='semibold'
    )

cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label("Accuracy (%)")

plt.tight_layout()
plt.show()

mejor_config = df_resultados.iloc[0]
print(f"
MEJOR CONFIGURACIÓN ENCONTRADA:")
print(f" • Nombre:          {mejor_config['Configuración']}")
print(f" • Capas Ocultas:   {mejor_config['Capas']}")
print(f" • Activación:      {mejor_config['Activación']}")
print(f" • Solver:          {mejor_config['Solver']}")
print(f" • Parámetros Tot.: {mejor_config['Parámetros']:,d} (Pesos: {mejor_config['Pesos']:,d}, Sesgos: {mejor_config['Sesgos']:,d})")
print(f" • Precisión:       {mejor_config['Accuracy']*100:.2f}%")


## 8. Entrenamiento del Modelo Definitivo con los Mejores Hiperparámetros

Configuramos el modelo óptimo con los parámetros más eficaces descubiertos durante la experimentación:
* `hidden_layer_sizes=(512, 256)`: Capacidad adecuada para representar interacciones complejas entre 1,184 características de visión.
* `activation='relu'`: Evita el desvanecimiento de gradiente y permite una convergencia ágil.
* `solver='adam'`: Tasa adaptativa ideal para optimización estocástica.
* `alpha=0.001`: Regularización L2 para suprimir el ruido en descriptores menos informativos.
* `early_stopping=True`: Con `n_iter_no_change=15` para monitorear la pérdida en un 10% reservado de validación y detenerse en el punto óptimo de generalización.
* `verbose=True`: Permite auditar la pérdida época tras época.


In [ ]:
# Instanciar el mejor MLPClassifier
mlp_optimo = MLPClassifier(
    hidden_layer_sizes=(512, 256),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=64,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=400,
    shuffle=True,
    random_state=SEED,
    tol=1e-4,
    verbose=True,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=15
)

print("Iniciando entrenamiento del modelo óptimo...")
t0 = time.time()
mlp_optimo.fit(X_train_scaled, y_train)
t1 = time.time()

# Desglose exhaustivo de parámetros del modelo óptimo
total_params, total_pesos, total_sesgos = contar_parametros_mlp(mlp_optimo)

print("="*65)
print("AUDITORÍA DE PARÁMETROS DEL MODELO ÓPTIMO MLP:")
print("="*65)
print(f" • Capa de Entrada -> Capa Oculta 1 (1184 -> 512):")
print(f"     Pesos: {mlp_optimo.coefs_[0].shape} = {mlp_optimo.coefs_[0].size:,} parámetros")
print(f"     Sesgos: {mlp_optimo.intercepts_[0].shape} = {mlp_optimo.intercepts_[0].size:,} parámetros")
print(f" • Capa Oculta 1 -> Capa Oculta 2 (512 -> 256):")
print(f"     Pesos: {mlp_optimo.coefs_[1].shape} = {mlp_optimo.coefs_[1].size:,} parámetros")
print(f"     Sesgos: {mlp_optimo.intercepts_[1].shape} = {mlp_optimo.intercepts_[1].size:,} parámetros")
print(f" • Capa Oculta 2 -> Capa de Salida (256 -> 10):")
print(f"     Pesos: {mlp_optimo.coefs_[2].shape} = {mlp_optimo.coefs_[2].size:,} parámetros")
print(f"     Sesgos: {mlp_optimo.intercepts_[2].shape} = {mlp_optimo.intercepts_[2].size:,} parámetros")
print("-"*65)
print(f" TOTAL PARÁMETROS ENTRENABLES: {total_params:,d}")
print(f" (Pesos sinápticos W: {total_pesos:,d} | Sesgos de neurona b: {total_sesgos:,d})")
print("="*65)

print(f"
Entrenamiento finalizado en {t1 - t0:.2f} segundos.")
print(f"Número total de iteraciones hasta convergencia: {mlp_optimo.n_iter_}")
print(f"Pérdida final de entrenamiento: {mlp_optimo.loss_:.5f}")


## 9. Curva de Pérdida y Convergencia del Entrenamiento

Visualizamos la evolución de la función de pérdida (`Loss Curve`) y la curva de precisión en validación durante las iteraciones de entrenamiento.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gráfico 1: Curva de pérdida
axes[0].plot(mlp_optimo.loss_curve_, color='#d62728', lw=2.5, label='Loss Entrenamiento')
axes[0].set_title("Curva de Pérdida (Loss Curve) durante las Épocas")
axes[0].set_xlabel("Iteraciones / Épocas")
axes[0].set_ylabel("Pérdida (Cross-Entropy Loss)")
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=10)

# Gráfico 2: Score de Validación durante Early Stopping
if hasattr(mlp_optimo, 'validation_scores_') and mlp_optimo.validation_scores_ is not None:
    axes[1].plot(mlp_optimo.validation_scores_, color='#1f77b4', lw=2.5, marker='o', markersize=4, label='Score de Validación')
    axes[1].set_title("Puntaje en Conjunto de Validación (Early Stopping)")
    axes[1].set_xlabel("Iteraciones / Épocas")
    axes[1].set_ylabel("Precisión de Validación")
    axes[1].grid(True, linestyle='--', alpha=0.6)
    axes[1].legend(fontsize=10)
else:
    axes[1].text(0.5, 0.5, "Early stopping no registró validation_scores_", ha='center', va='center')

plt.tight_layout()
plt.show()


## 10. Evaluación Exhaustiva en el Conjunto de Prueba


In [ ]:
# Predicciones sobre el conjunto de test
y_pred = mlp_optimo.predict(X_test_scaled)

# Cálculo de métricas globales
acc_test = accuracy_score(y_test, y_pred)
print("="*60)
print(f"PRECISIÓN GLOBAL DEL MODELO ÓPTIMO (ACCURACY): {acc_test * 100:.2f}%")
print("="*60)

# Reporte detallado de clasificación con nombres de clases reales
reporte_texto = classification_report(y_test, y_pred, target_names=clases_nombres)
print("
REPORTE DE CLASIFICACIÓN DETALLADO:
")
print(reporte_texto)


## 11. Visualizaciones Detalladas del Desempeño: Matriz de Confusión y Métricas por Clase

Generamos la matriz de confusión normalizada para detectar similitudes entre vehículos (como `SEDAN` vs `Hatchback` o `Truck` vs `Pickup`), junto a un gráfico de barras desglosando **Precision**, **Recall** y **F1-Score** por cada vehículo.


In [ ]:
# 1. Matriz de confusión normalizada (porcentual por clase real)
cm = confusion_matrix(y_test, y_pred)
cm_normalizada = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# 2. Desglose de métricas por clase
precision, recall, f1, soporte = precision_recall_fscore_support(y_test, y_pred, average=None)

fig = plt.figure(figsize=(18, 7))

# Subgráfico 1: Heatmap de la Matriz de Confusión
ax1 = fig.add_subplot(1, 2, 1)
sns.heatmap(
    cm_normalizada,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=clases_nombres,
    yticklabels=clases_nombres,
    cbar=True,
    ax=ax1,
    linewidths=0.5
)
ax1.set_title("Matriz de Confusión Normalizada (Tasa de Acierto por Clase)", fontsize=12, fontweight='bold')
ax1.set_xlabel("Clase Predicha", fontsize=11)
ax1.set_ylabel("Clase Verdadera", fontsize=11)
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', rotation=0)

# Subgráfico 2: Métricas Precision, Recall y F1 por Clase
ax2 = fig.add_subplot(1, 2, 2)
ancho_barra = 0.25
indices = np.arange(len(clases_nombres))

ax2.bar(indices - ancho_barra, precision, width=ancho_barra, label='Precision', color='#3498db', edgecolor='black', alpha=0.85)
ax2.bar(indices,               recall,    width=ancho_barra, label='Recall',    color='#2ecc71', edgecolor='black', alpha=0.85)
ax2.bar(indices + ancho_barra, f1,        width=ancho_barra, label='F1-Score',  color='#e67e22', edgecolor='black', alpha=0.85)

ax2.set_title("Métricas de Clasificación por Categoría de Vehículo", fontsize=12, fontweight='bold')
ax2.set_xlabel("Tipo de Vehículo", fontsize=11)
ax2.set_ylabel("Puntaje (0.0 - 1.0)", fontsize=11)
ax2.set_xticks(indices)
ax2.set_xticklabels(clases_nombres, rotation=45, ha='right')
ax2.set_ylim(0, 1.08)
ax2.grid(axis='y', linestyle='--', alpha=0.6)
ax2.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()


## 12. Interpretabilidad: ¿Qué Descriptores Visuales Tienen Mayor Impacto en la Red Neuronal?

Podemos analizar los pesos sinápticos de la **primera capa oculta** ($W^{(1)} \in \mathbb{R}^{1184 \times 512}$) del MLP. Calculando la norma o magnitud promedio de los pesos correspondientes a cada familia de descriptores (`Canny`, `LBP`, `DCT`, `ORB`, `HSV`, `Hu`, `LoG`, `Harris`), descubrimos qué características visuales utiliza la red con mayor intensidad para distinguir las clases de vehículos.


In [ ]:
# Extraer la matriz de pesos de la primera capa oculta (1184 características x 512 neuronas)
pesos_capa_1 = mlp_optimo.coefs_[0]

# Calcular la importancia media como la norma L2 promedio de los pesos de cada entrada
importancia_features = np.linalg.norm(pesos_capa_1, axis=1)

# Agrupar importancia por familia de descriptor
importancia_por_descriptor = {}
conteo_por_descriptor = {}

for col_idx, col_name in enumerate(df.columns[:-1]):
    prefijo = col_name.split('_')[0].upper()
    importancia_por_descriptor[prefijo] = importancia_por_descriptor.get(prefijo, 0.0) + importancia_features[col_idx]
    conteo_por_descriptor[prefijo] = conteo_por_descriptor.get(prefijo, 0) + 1

# Promediar por el número de características de cada descriptor para una comparación justa
importancia_promedio = {
    k: importancia_por_descriptor[k] / conteo_por_descriptor[k]
    for k in importancia_por_descriptor
}

df_importancia = pd.DataFrame({
    'Descriptor': list(importancia_promedio.keys()),
    'Magnitud_Promedio': list(importancia_promedio.values()),
    'Num_Features': [conteo_por_descriptor[k] for k in importancia_promedio]
}).sort_values(by='Magnitud_Promedio', ascending=False)

# Graficar la importancia relativa de descriptores
fig, ax = plt.subplots(figsize=(12, 5))
colores_imp = sns.color_palette("rocket_r", len(df_importancia))
barras_imp = ax.bar(df_importancia['Descriptor'], df_importancia['Magnitud_Promedio'], color=colores_imp, edgecolor='black', alpha=0.85)

ax.set_title("Impacto Relativo de cada Familia de Descriptores de Visión en la Red MLP", fontsize=13, fontweight='bold')
ax.set_xlabel("Descriptor de Visión Artificial")
ax.set_ylabel(r"Magnitud Promedio de Pesos Sinápticos ($||W^{(1)}||$)")
ax.grid(axis='y', linestyle='--', alpha=0.6)

for bar in barras_imp:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f"{yval:.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("Resumen de importancia promedio por descriptor:")
print(df_importancia.to_string(index=False))


## 13. Conclusiones y Comparativa: Featuring vs Flattening

### Resumen de Hallazgos y Parámetros:
1. **Poder Discriminativo de Descriptores de Visión:** El uso de características visuales estructuradas (`Canny`, `LBP`, `DCT`, `ORB`, `HSV`, etc.) proporciona al perceptrón multicapa información de alto nivel (bordes, frecuencias, texturas y color), permitiendo una mejor representación que los píxeles directos planos.
2. **Impacto Decisivo del Escalado (`StandardScaler`):** Dado que los momentos Hu y descriptores de frecuencia difieren en varios órdenes de magnitud con respecto a los bordes de Canny, la estandarización garantiza que el optimizador Adam converja rápidamente sin gradientes desbordados.
3. **Auditoría de Parámetros:**
   * La arquitectura óptima bicapa `(512, 256)` cuenta con **740,618 parámetros entrenables** ($606,208 + 131,072 + 2,560$ pesos y $512 + 256 + 10$ sesgos).
   * Este volumen de parámetros ofrece el equilibrio ideal entre capacidad representacional y prevención de sobreajuste cuando se combina con regularización L2 (`alpha=0.001`).
4. **Hiperparámetros Óptimos Identificados:**
   * **Arquitectura:** Red profunda de dos capas `(512, 256)` ofrece una excelente capacidad de abstracción no lineal.
   * **Activación:** `relu` superó consistentemente a `logistic` y `tanh` tanto en rapidez de convergencia como en precisión final.
   * **Solver:** `adam` demostró ser superior y más estable que `sgd`.
   * **Regularización (`alpha=0.001`):** Fundamental para controlar la complejidad sobre un espacio de entrada de 1,184 dimensiones.
   * **Early Stopping:** Previno el sobreajuste deteniendo el entrenamiento en el pico máximo de rendimiento sobre validación.
